<a href="https://colab.research.google.com/github/amosagekouassi-source/DI-Bootcamp/blob/master/Dailychallenge_J3_W6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 1: Understanding BERT and XLM-RoBERTa

### BERT (Bidirectional Encoder Representations from Transformers)
BERT est conçu pour pré-entraîner des représentations bidirectionnelles profondes à partir de texte non étiqueté en conditionnant conjointement le contexte gauche et droit dans toutes les couches.

### XLM-RoBERTa
Il s'agit d'une version multilingue de RoBERTa entraînée sur un ensemble de données massif dans 100 langues différentes. Contrairement à BERT, il n'utilise pas l'objectif de prédiction de la phrase suivante (NSP).

**Tokenization** : Ces modèles utilisent des algorithmes comme WordPiece (BERT) ou SentencePiece (XLM-R) pour segmenter le texte en unités de sous-mots, ce qui permet de gérer efficacement les mots rares et le vocabulaire multilingue.

### 2. Tokenizing Text
Objectif: Utiliser `BertTokenizer` et `XLMRobertaTokenizer` pour explorer les `input_ids` et `attention_mask`.

In [16]:
from transformers import BertTokenizer, XLMRobertaTokenizer

# Initialisation des tokenizers
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')
xlm_tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')

# Extraction des phrases pour test
sentence1 = train_df.premise.values[0]
sentence2 = train_df.hypothesis.values[0]

# Tâche 2: Tokenisation utilisant l'appel direct (recommandé)
tokens = bert_tokenizer(
    sentence1,
    sentence2,
    add_special_tokens=True
)

print("Input IDs:", tokens['input_ids'])
print("Attention Mask:", tokens['attention_mask'])
print("Décodage:", bert_tokenizer.decode(tokens['input_ids']))

Input IDs: [101, 89451, 10176, 10231, 42621, 15240, 10341, 119, 102, 140, 112, 10176, 10231, 42621, 15240, 10341, 119, 102]
Attention Mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Décodage: [CLS] Ceci est une prémisse. [SEP] C ' est une prémisse. [SEP]


### 3. Preparing Input Data for the Model
Objectif: Gérer le padding, la truncation et explorer le vocabulaire.

In [17]:
# Tâche 3: Exploration et préparation
print(f"BERT Vocab size: {bert_tokenizer.vocab_size}")
print(f"BERT Special Tokens Map: {bert_tokenizer.special_tokens_map}")

# Padding et Truncation via l'appel direct
final_input = bert_tokenizer(
    sentence1,
    sentence2,
    max_length=50,
    padding='max_length',
    truncation=True,
    add_special_tokens=True
)

print("\nPadding appliqué (IDs):", final_input['input_ids'])
print("Décodage du padding:", bert_tokenizer.decode(final_input['input_ids']))

BERT Vocab size: 119547
BERT Special Tokens Map: {'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}

Padding appliqué (IDs): [101, 89451, 10176, 10231, 42621, 15240, 10341, 119, 102, 140, 112, 10176, 10231, 42621, 15240, 10341, 119, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Décodage du padding: [CLS] Ceci est une prémisse. [SEP] C ' est une prémisse. [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]


### Mise en pratique : Fonction d'encodage pour le Dataset
Cette étape automatise la préparation des données en utilisant les concepts validés dans les tâches 2 (tokenisation) et 3 (padding/truncation) pour traiter l'intégralité du DataFrame.

In [18]:
def encode_data(df, tokenizer, max_len=128):
    # Tokenisation des paires (premise, hypothesis)
    # L'appel direct gère automatiquement les listes de paires
    encoded = tokenizer(
        df['premise'].tolist(),
        df['hypothesis'].tolist(),
        add_special_tokens=True,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='np'
    )
    return encoded['input_ids'], encoded['attention_mask']

# Exemple de préparation pour BERT sur le premier pli
train_fold = train_df[train_df['fold'] != 0]
val_fold = train_df[train_df['fold'] == 0]

X_train_ids, X_train_mask = encode_data(train_fold, bert_tokenizer)
X_val_ids, X_val_mask = encode_data(val_fold, bert_tokenizer)

print(f"Forme des IDs d'entraînement : {X_train_ids.shape}")
print(f"Exemple de décodage : {bert_tokenizer.decode(X_train_ids[0])}")

Forme des IDs d'entraînement : (24, 128)
Exemple de décodage : [CLS] Le chat dort. [SEP] Le chat ne dort pas. [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]


### 4. Loading and Exploring the Dataset
Objectif: Charger les données et identifier les colonnes utiles.

In [9]:
import pandas as pd
import numpy as np
import os

# Création de données factices si les fichiers n'existent pas
if not os.path.exists('/content/train.csv'):
    data = {
        'premise': ["Ceci est une prémisse.", "Le chat dort.", "Il fait beau."] * 10,
        'hypothesis': ["C'est une prémisse.", "Le chat ne dort pas.", "Le temps est clair."] * 10,
        'label': [0, 2, 0] * 10
    }
    train_df = pd.DataFrame(data)
    train_df.to_csv('/content/train.csv', index=False)

train_df = pd.read_csv('/content/train.csv')
print(f"Dimensions : {train_df.shape}")
display(train_df.head())

Dimensions : (30, 3)


,premise,hypothesis,label
0,Ceci est une prémisse.,C'est une prémisse.,0
1,Le chat dort.,Le chat ne dort pas.,2
2,Il fait beau.,Le temps est clair.,0
3,Ceci est une prémisse.,C'est une prémisse.,0
4,Le chat dort.,Le chat ne dort pas.,2


### 5. Creating Cross-Validation Folds
Objectif: Créer 5 plis stratifiés.

In [12]:
from sklearn.model_selection import StratifiedKFold

# Configuration de la validation croisée à 5 plis
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Initialisation de la colonne fold
train_df['fold'] = -1

# Attribution des plis en fonction du label pour maintenir la distribution
for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df.label)):
    train_df.loc[val_idx, 'fold'] = fold

print("Distribution des labels par fold (0, 1, 2) :")
print(train_df.groupby(['fold', 'label']).size().unstack())
print("\nStructure du dataset avec les plis :")
display(train_df.head())

Distribution des labels par fold (0, 1, 2) :
label  0  2
fold       
0      4  2
1      4  2
2      4  2
3      4  2
4      4  2

Structure du dataset avec les plis :


,premise,hypothesis,label,fold
0,Ceci est une prémisse.,C'est une prémisse.,0,0
1,Le chat dort.,Le chat ne dort pas.,2,2
2,Il fait beau.,Le temps est clair.,0,4
3,Ceci est une prémisse.,C'est une prémisse.,0,3
4,Le chat dort.,Le chat ne dort pas.,2,1
